In [9]:
model = "llama3.2:1b"

#### Task 1: Simple Chain with Retrieval

**Objective:**

Implement a simple RAG chain with ChatOllama, HuggingFaceEmbeddings and Chroma. 

Process: 

1. Retrieve documents from chroma db based on query
2. Invoke chain with retrieved documents as input

**Task Description:**

- load llm model via ollama
- load embedding model via ollama with `ollama pull pull bge-m3` (if not yet done)
- create chroma db client
- create prompt template for summarization
- create simple chain with following steps: retrieved documents, prompt, model, output parser
- create query and perform similarity search with a query
- invoke chain and pass retrieved documents to the chain


**Useful links:**

- [RAG with Ollama](https://python.langchain.com/v0.2/docs/tutorials/local_rag/)
- [Streaming in Langchain](https://python.langchain.com/docs/concepts/streaming/)


In [10]:
from langchain_ollama import ChatOllama

# ADD HERE YOUR CODE
model = ChatOllama(model=model)

In [11]:
from langchain_ollama import OllamaEmbeddings

# ADD HERE YOUR CODE
embedding_model = OllamaEmbeddings(model="bge-m3")

In [12]:
from langchain_chroma import Chroma
import chromadb
import chromadb
from chromadb.config import DEFAULT_TENANT, DEFAULT_DATABASE, Settings

client = chromadb.HttpClient(
    host="localhost",
    port=8000,
    ssl=False,
    headers=None,
    settings=Settings(allow_reset=True, anonymized_telemetry=False),
    tenant=DEFAULT_TENANT,
    database=DEFAULT_DATABASE,
)

# Create a collection
# ADD HERE YOUR CODE
collection = client.get_or_create_collection("ai_model_book")

# Create chromadb
# ADD HERE YOUR CODE
vector_db_from_client = Chroma(collection_name="ai_model_book", embedding_function=embedding_model, client=client)

In [13]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    "Summarize the main themes in these retrieved docs: {docs}"
)


# Convert loaded documents into strings by concatenating their content
# and ignoring metadata
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


chain = {"docs": format_docs}|prompt|model|StrOutputParser()

In [14]:
search_query = "Types of Machine Learning Systems"

# ADD HERE YOUR CODE
# Perform vector search
docs = vector_db_from_client.similarity_search(search_query)

print(docs)

[Document(metadata={'page': 33, 'source': './AI_Book.pdf'}, page_content='Types of Machine Learning Systems\nThere are so many different types of Machine Learning systems that it is useful to\nclassify them in broad categories based on:\n Whether or not they are trained with human supervision (supervised, unsuper\nvised, semisupervised, and Reinforcement Learning)\n Whether or not they can learn incrementally on the fly (online versus batch\nlearning)\n Whether they work by simply comparing new data points to known data points,\nor instead detect patterns in the training data and build a predictive model, much\nlike scientists do (instance-based versus model-based learning)\nThese criteria are not exclusive; you can combine them in any way you like. For\nexample, a state-of-the-art spam filter may learn on the fly using a deep neural net\nwork model trained using examples of spam and ham; this makes it an online, model-\nbased, supervised learning system.\nLets look at each of these cr

In [15]:
chain.invoke(docs)

"Based on the retrieved documents, some main themes that emerge are:\n\n1. **Classification and Categorization**: The authors discuss how to categorize Machine Learning (ML) systems into broad categories based on training methods, including supervised, unsupervised, semisupervised, and reinforcement learning.\n2. **Learning Methods**: The document explains the different types of ML systems that can be classified based on their learning methods:\n\t* Supervised learning: using labeled data to train models\n\t* Instance-based vs model-based learning: instance-based involves learning from individual examples, while model-based involves learning a general model for new instances.\n\t* Incremental learning: able to learn from incoming data incrementally without retraining the entire system.\n3. **Generalization**: The document emphasizes the importance of generalization in ML tasks, where systems need to perform well on unseen data.\n4. **Monitoring and Rebalancing**: Regular monitoring and

In [16]:
# Simple stream the chain output
for chunk in chain.stream(docs):
    print(chunk, end="", flush=True)

Based on the retrieved documents, the main themes in these types of machine learning systems are:

1. **Classification and Prediction**: The system is designed to classify data into predefined categories or make predictions based on input data.
2. **Supervised, Unsupervised, Semisupervised, and Reinforcement Learning**: Machine learning systems can be categorized into four main types:
	* Supervised learning: Where the training data includes desired solutions (labels) and the system learns to predict these labels.
	* Unsupervised learning: Where the system identifies patterns in the data without any predefined labels.
	* Semisupervised learning: A combination of supervised and unsupervised learning where the system uses both labeled and unlabeled data.
	* Reinforcement learning: A type of machine learning where the system learns by trial and error, receiving feedback in the form of rewards or penalties for its actions.
3. **Instance-Based Learning vs. Model-Based Learning**: Two main ap

In [17]:
# More complex async event streaming
async for event in chain.astream_events(docs, version="v2"):
    kind = event["event"]
    if kind == "on_chat_model_stream":
        print(event["data"]["chunk"].content, end="", flush=True)

C:\Users\josip\AppData\Local\Temp\ipykernel_14816\1927557568.py:2: LangChainBetaWarning: This API is in beta and may change in the future.
  async for event in chain.astream_events(docs, version="v2"):


Based on the retrieved documents, here are the main themes that emerge:

1. **Classification**: The document highlights the importance of classifying Machine Learning systems into broad categories based on their characteristics, such as supervised/unsupervised learning, online/offline learning, instance-based/model-based learning, and batch/online learning.
2. **Supervision**: The classification is further divided into four major categories: supervised/unsupervised learning, semisupervised learning, Reinforcement Learning (RL), and model-based learning. Each category requires different approaches to generalization.
3. **Learning Paradigms**: The document distinguishes between two main paradigms:
	* Instance-based learning (e.g., spam filtering): learns from examples by heart and then generalizes to new cases using a similarity measure.
	* Model-based learning (e.g., AlphaGo): learns from data by analyzing patterns and applying a policy learned through trial-and-error.
4. **Generalizati

#### Task 2: Q&A with RAG

**Objective:**

Implement a Q/A retrieval chain with ChatOllama, HuggingFaceEmbeddings and Chroma

**Task Description:**

- create RAG-Q/A prompt template
- create retriever from vector db client (instead of manually passing in docs, we automatically retrieve them from our vector store based on the user question)
- create simple chain with following steps: retriever, formatting retrieved docs, user question, prompt, model, output parser
- create question for Q/A retrieval chain
- invoke chain and with question

**Useful links:**

- [RAG with Ollama](https://python.langchain.com/v0.2/docs/tutorials/local_rag/)

In [18]:
from langchain_core.runnables import RunnablePassthrough

prompt_template = """
You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.

<context>
{context}
</context>

Answer the following question:

{question}"""

# ADD HERE YOUR CODE
rag_prompt = ChatPromptTemplate.from_template(prompt_template)

# ADD HERE YOUR CODE
retriever = vector_db_from_client.as_retriever()

# ADD HERE YOUR CODE
qa_rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | model
    | StrOutputParser()
)

In [19]:
qa_rag_chain

{
  context: VectorStoreRetriever(tags=['Chroma', 'OllamaEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x0000012BFBA39A90>)
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], template="\nYou are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\n\n<context>\n{context}\n</context>\n\nAnswer the following question:\n\n{question}"))])
| ChatOllama(model='llama3.2:1b', _client=<ollama._client.Client object at 0x0000012BFDB52510>, _async_client=<ollama._client.AsyncClient object at 0x0000012BFF1D6110>)
| StrOutputParser()

In [20]:
question = "What is supervised learning?"

# ADD HERE YOUR CODE
qa_rag_chain.invoke(question)

'Supervised learning refers to a type of machine learning where the training data includes labeled or classified examples, and the algorithm learns to make predictions or decisions based on this labeled data. The goal of supervised learning is for the algorithm to learn from the data without being explicitly told how to do it, by making educated guesses about the input data that correspond with the expected output labels.'

In [21]:
# More complex async event streaming
async for event in qa_rag_chain.astream_events(question, version="v2"):
    kind = event["event"]
    if kind == "on_chat_model_stream":
        print(event["data"]["chunk"].content, end="", flush=True)

Supervised learning is a type of machine learning where the training data includes labeled examples, and the algorithm tries to learn without a teacher by predicting the correct labels for new, unseen data. The system learns from the relationships between input features (X) and output labels (y).

#### Alternative: Using pre-built ConversationalRetrievalChain Class

In [22]:
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory

In [23]:
retriever = vector_db_from_client.as_retriever()
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

In [24]:
qa_chain = ConversationalRetrievalChain.from_llm(
    model, retriever=retriever, memory=memory, verbose=False
)

In [25]:
# More complex async event streaming
async for event in qa_chain.astream_events("What is supervised learning?", version="v2"):
    kind = event["event"]
    if kind == "on_chat_model_stream":
        print(event["data"]["chunk"].content, end="", flush=True)

Supervised learning is a type of machine learning where the algorithm is trained on labeled data, meaning that each example in the training set has a corresponding label or output that corresponds to its input.

In other words, in supervised learning, you have:

* A dataset with multiple examples, each with an associated target variable (also known as the outcome or label)
* A model that tries to predict the value of the target variable for new, unseen examples
* The goal is to train the model on labeled data so it can make accurate predictions on new data

Here's an example:

Suppose you want to build a system that can classify images of cats and dogs as either "cat" or "dog". You have a dataset with many images of cats and dogs, along with their corresponding labels (e.g. cat, dog, etc.). The goal is to train a model that can predict the label for new, unseen images.

In supervised learning, you would:

1. Collect and preprocess the labeled data
2. Train the model on the labeled data

In [26]:
# More complex async event streaming
async for event in qa_chain.astream_events("Which algorithms can be used there?", version="v2"):
    kind = event["event"]
    if kind == "on_chat_model_stream":
        print(event["data"]["chunk"].content, end="", flush=True)

Here is a rephrased version of the follow up question:

Can unsupervised learning algorithms also be applied to datasets where each example has an associated target variable or output, without relying on external labels?Yes, some unsupervised learning algorithms can be applied to datasets where each example has an associated target variable or output. 

For example, one type of algorithm is called a Clustering algorithm, which groups similar data points together based on their features and patterns in the data. This can be useful for tasks such as:

* Identifying customer segments based on purchase behavior
* Grouping similar products together based on characteristics such as price, quality, or brand
* Detecting anomalies or outliers in a dataset

Another type of algorithm is called a Semi-Supervised Learning algorithm, which can handle datasets where some of the data points have corresponding labels (i.e., output), while others do not. In this case, the algorithms can use the unlabele